# ST-GNN Final Model and Evaluation (Strict Spatial Holdout)

- **Features:** mean_pr, mean_tair, Mean Elevation, mean_swe_lag_7
- **Training HUC12s:** 495 (original 535 minus DHSVM ∪ LIDAR overlap)
- **Test HUC12s:** DHSVM (92) + LIDAR (39) = 131
- **Adjacency:** A_train = train–train only; A_test = test–test only. No cross edges.
- **5 seeds:** train 5 models; report mean (and std) of metrics; save all 5; test uses mean prediction.

In [ ]:
import os
import boto3
import pandas as pd
import geopandas as gpd
import numpy as np
import mlflow
import mlflow.pytorch
import torch
import random
import zarr
from snowML.datapipe.utils import data_utils as du
from snowML.datapipe.utils import get_geos as gg

import importlib
import train_eval_pipeline_spatial_holdout as tef
importlib.reload(tef)
from train_eval_pipeline_spatial_holdout import train_model, run_test_predictions

## Config

In [3]:
BUCKET_NAME = "snowml-shape"
MODEL_READY_BUCKET = "snowml-model-ready-stgnn"
AWS_REGION = "us-west-2"

HUC_LEVEL = "12"
HUC8_IDS = [
    17020009, 17020011, 17110005, 17110006, 17110009,
    17010304, 17010302, 17060207, 17060208, 17030001, 17030002,
    17110008, 17030003, 17020010, 17110007,
]
EXCLUDED_HUCS_CA = [
    "171100050101", "171100050102", "171100050201", "171100050202", "171100050203",
    "171100050301", "171100050302", "171100050303", "171100050304", "171100050305",
    "171100050306", "171100050401",
]
WEIRD_HUCS = ['171100060305', '171100060401', '171100060404', '171100060405']
EXCLUDED_HUCS = EXCLUDED_HUCS_CA + WEIRD_HUCS

ADJ_SIGMA_CUTOFF = 3
ADJ_ALPHA = 0.7

FINAL_DYNAMIC = ["mean_pr", "mean_tair", "mean_swe_lag_7"]
FINAL_STATIC_MR = ["Mean Elevation"]

VAL_SPLIT = 0.2
NUM_EPOCHS = 20
BATCH_SIZE = 32
SEQ_LEN = 30
EARLY_STOPPING = True
PATIENCE = 5
REL_THRESHOLD = 0.03
SEEDS = [41, 42, 43, 44, 45]

MLFLOW_TRACKING_URI = "arn:aws:sagemaker:us-west-2:677276086662:mlflow-tracking-server/wi26-snowml-mlflow"
EXPERIMENT_NAME = "stgnn_final_model"

# DHSVM and LIDAR HUC12 lists from dhsvm_lidar_hucs.csv (basin == "lidar" → LIDAR, else → DHSVM)
dhsvm_lidar_path = "../Dataprep/dhsvm_lidar_hucs.csv"

In [4]:
s3_client = boto3.client("s3", region_name=AWS_REGION)

## Load LSTM HUC12s and define train / test split

In [5]:
def get_geo_jsons(huc_ids, huc_level, bucket_nm="snowml-shape"):
    gdf_list = []
    for huc_id in huc_ids:
        f_out = f"Huc{huc_level}_in_{huc_id}.geojson"
        if du.isin_s3(bucket_nm, f_out):
            temp = du.s3_to_gdf(bucket_nm, f_out)
        else:
            try:
                temp = gg.get_geos(huc_id, huc_level, s3_save=True, bucket_nm=bucket_nm)
            except Exception as e:
                print(f"Error getting geos for {huc_id}: {e}")
                continue
        if temp is not None:
            temp['huc8'] = huc_id
            gdf_list.append(temp)
    if not gdf_list:
        return gpd.GeoDataFrame()
    return gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True))


def get_model_ready_data(huc_ids, bucket_nm):
    huc_dfs = []
    for huc_id in huc_ids:
        f_out = f"model_ready_huc{huc_id}.csv"
        df = du.s3_to_df(f_out, bucket_nm)
        df['huc_id'] = huc_id
        df['day'] = pd.to_datetime(df["day"])
        huc_dfs.append(df)
    return pd.concat(huc_dfs, ignore_index=True)

In [6]:
# LSTM 535 HUC12s: geometry from HUC8 regions, exclude list applied, huc_id as string
gdf_lstm = get_geo_jsons(HUC8_IDS, HUC_LEVEL, BUCKET_NAME)
gdf_lstm["huc_id"] = gdf_lstm["huc_id"].astype(str)
gdf_lstm = gdf_lstm[~gdf_lstm["huc_id"].isin(EXCLUDED_HUCS)]
lstm_huc12s = gdf_lstm["huc_id"].tolist()
print(f"LSTM HUC12s: {len(lstm_huc12s)}")

LSTM HUC12s: 535


### DHSVM / LIDAR list and train vs test HUC12s

Load test HUC12s from `dhsvm_lidar_hucs.csv` (basin `"lidar"` → LIDAR, else → DHSVM).  
Training HUC12s = LSTM 535 minus (DHSVM ∪ LIDAR). Order: **train first, then test** in `unique_huc12s`.

In [7]:
dhsvm_lidar = pd.read_csv(dhsvm_lidar_path)
dhsvm_lidar["huc_id"] = dhsvm_lidar["huc_id"].astype(str)
LIDAR_HUC12S = dhsvm_lidar.loc[dhsvm_lidar["basin"] == "lidar", "huc_id"].tolist()
DHSVM_HUC12S = dhsvm_lidar.loc[dhsvm_lidar["basin"] != "lidar", "huc_id"].tolist()

test_huc12s_set = set(DHSVM_HUC12S) | set(LIDAR_HUC12S)
training_huc12s = [h for h in lstm_huc12s if h not in test_huc12s_set]
test_huc12s = list(test_huc12s_set)
unique_huc12s = training_huc12s + test_huc12s

n_train = len(training_huc12s)
n_test = len(unique_huc12s) - n_train
train_node_idx = np.arange(n_train)
test_node_idx = np.arange(n_train, len(unique_huc12s))
print(f"Unique HUC12s: {len(unique_huc12s)} (train={n_train}, test={n_test})")

Unique HUC12s: 626 (train=495, test=131)


### Full geometry (train + test) and node order

Combine train and test geometries, then reindex to `unique_huc12s` so node order is consistent everywhere.

In [9]:
gdf_train = gdf_lstm[gdf_lstm["huc_id"].isin(training_huc12s)]
gdf_test = get_geo_jsons(test_huc12s, HUC_LEVEL, BUCKET_NAME)
gdf_test["huc_id"] = gdf_test["huc_id"].astype(str)

cols_needed = ["huc_id", "geometry"]
gdf_full = pd.concat([gdf_train[cols_needed], gdf_test[cols_needed]], ignore_index=True)
gdf_full = gdf_full.set_index("huc_id").reindex(unique_huc12s).reset_index()
assert gdf_full["geometry"].isna().sum() == 0, "Some HUC12 geometries are missing after reindex"
print(f"gdf_full: {len(gdf_full)} HUC12s (train={n_train}, test={n_test})")

gdf_full: 626 HUC12s (train=495, test=131)


In [10]:
huc_dfs = get_model_ready_data(unique_huc12s, MODEL_READY_BUCKET)
huc_dfs = huc_dfs.dropna(subset=['mean_swe_lag_7']).copy()
print(f"huc_dfs: {len(huc_dfs)} rows")

huc_dfs: 8912362 rows


In [11]:
huc_dfs

,day,mean_pr,mean_tair,mean_vs,mean_srad,mean_hum,mean_swe,Mean Elevation,Predominant Snow,Mean Forest Cover,mean_swe_lag_7,mean_swe_lag_30,mean_swe_lag_60,snow_cover,era5_swe,slope,huc_id,snodas_swe
7,1983-10-08,0.000000,2.412500,4.175000,101.750000,0.511875,0.0,1758.266724,Maritime,38.389219,0.000333,NaN,NaN,0.005798,0.000087,27.295113,170200090101,NaN
8,1983-10-09,0.000000,1.637500,3.100000,162.150000,0.648375,0.0,1758.266724,Maritime,38.389219,0.000000,NaN,NaN,0.004234,0.000063,27.295113,170200090101,NaN
9,1983-10-10,0.000000,-0.050000,2.825000,165.600000,0.673125,0.0,1758.266724,Maritime,38.389219,0.000000,NaN,NaN,0.008129,0.000124,27.295113,170200090101,NaN
10,1983-10-11,0.000000,3.250000,1.425000,159.450000,0.540125,0.0,1758.266724,Maritime,38.389219,0.000000,NaN,NaN,0.007820,0.000116,27.295113,170200090101,NaN
11,1983-10-12,0.425000,4.850000,2.575000,136.800000,0.475875,0.0,1758.266724,Maritime,38.389219,0.000000,NaN,NaN,0.004607,0.000068,27.295113,170200090101,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8916739,2022-09-25,0.000000,19.104545,3.809091,196.472727,0.643045,0.0,142.591232,Ephemeral,34.804069,0.000000,0.0,0.0,NaN,NaN,NaN,171100130302,NaN
8916740,2022-09-26,0.000000,21.322727,2.063636,193.900000,0.587091,0.0,142.591232,Ephemeral,34.804069,0.000000,0.0,0.0,NaN,NaN,NaN,171100130302,NaN
8916741,2022-09-27,0.000000,19.400000,2.563636,190.727273,0.653773,0.0,142.591232,Ephemeral,34.804069,0.000000,0.0,0.0,NaN,NaN,NaN,171100130302,NaN
8916742,2022-09-28,0.763636,15.777273,2.818182,103.027273,0.709864,0.0,142.591232,Ephemeral,34.804069,0.000000,0.0,0.0,NaN,NaN,NaN,171100130302,NaN


## Time series tensors and normalization (train period + train nodes only)

In [12]:
def create_dynamic_features(huc_dfs, huc_ids, feature_names):
    huc_groups = huc_dfs.groupby('huc_id')['day'].apply(lambda x: set(x.astype(str)))
    common_timestamps = set.intersection(*huc_groups)
    master_index = sorted(common_timestamps)
    all_data, target_data = [], []
    for huc in huc_ids:
        df = huc_dfs[huc_dfs['huc_id'] == huc].copy()
        df['day'] = df['day'].astype(str)
        df = df.set_index('day').reindex(master_index)
        all_data.append(df[feature_names].values.T.astype(float))
        target_data.append(df['mean_swe'].values.astype(float))
    dynamic = torch.tensor(np.array(all_data), dtype=torch.float32)
    target = torch.tensor(np.array(target_data), dtype=torch.float32)
    return dynamic, target, master_index

dynamic_full, target_tensor, master_index = create_dynamic_features(
    huc_dfs, unique_huc12s, feature_names=FINAL_DYNAMIC
)
num_timesteps = dynamic_full.shape[2]
split_idx = int(num_timesteps * (1 - VAL_SPLIT))

dyn_train_slice = dynamic_full[train_node_idx, :, :split_idx].numpy()
dyn_mean = np.nanmean(dyn_train_slice, axis=(0, 2))
dyn_std = np.nanstd(dyn_train_slice, axis=(0, 2))
dyn_std = np.where(np.isfinite(dyn_std) & (dyn_std > 0), dyn_std, 1.0)
dynamic_full = (dynamic_full - torch.tensor(dyn_mean, dtype=torch.float32)[None, :, None]) / torch.tensor(dyn_std, dtype=torch.float32)[None, :, None]

print(f"dynamic_full {dynamic_full.shape}, timesteps={num_timesteps}, split_idx={split_idx}")

dynamic_full torch.Size([626, 3, 14237]), timesteps=14237, split_idx=11389


## Adjacency: A_base, A_train (train-train only), A_test (test-test only)

In [13]:
def build_huc12_adjacency_hybrid(gdf, epsg_project=32610, alpha=ADJ_ALPHA, prox_cutoff_sigma=ADJ_SIGMA_CUTOFF,
                                 add_self_loops=True, normalize_adj=True, eps=1e-12):
    g = gdf[["huc_id", "geometry"]].copy().to_crs(epsg=epsg_project).reset_index(drop=True)
    N = len(g)
    geoms = g.geometry.values
    perim = np.array([geom.length for geom in geoms], dtype=float)
    area = np.array([geom.area for geom in geoms], dtype=float)
    centroids = g.geometry.centroid
    coords = np.array([[c.x, c.y] for c in centroids], dtype=float)
    diff = coords[:, None, :] - coords[None, :, :]
    d_mat = np.linalg.norm(diff, axis=2)
    np.fill_diagonal(d_mat, 0.0)
    d_nonzero = d_mat[d_mat > 0]
    sigma = float(np.median(d_nonzero) + eps)
    cutoff = prox_cutoff_sigma * sigma
    w_prox = np.exp(-(d_mat ** 2) / (2.0 * sigma ** 2))
    w_prox[d_mat > cutoff] = 0.0
    np.fill_diagonal(w_prox, 0.0)
    w_shared = np.zeros((N, N), dtype=float)
    sindex = g.sindex
    for i in range(N):
        geom_i = geoms[i]
        b_i = geom_i.boundary
        cand = list(sindex.intersection(geom_i.bounds))
        for j in cand:
            if j <= i:
                continue
            geom_j = geoms[j]
            if not geom_i.touches(geom_j):
                continue
            shared = b_i.intersection(geom_j.boundary)
            L = float(shared.length)
            wij = (2.0 * L) / (perim[i] + perim[j] + eps)
            w_shared[i, j] = w_shared[j, i] = wij
    A_raw = alpha * w_shared + (1.0 - alpha) * w_prox
    A_used = A_raw.copy()
    if add_self_loops:
        np.fill_diagonal(A_used, A_used.diagonal() + 1.0)
    if normalize_adj:
        d = A_used.sum(axis=1)
        D_inv_sqrt = np.diag(1.0 / np.sqrt(d + eps))
        A_used = D_inv_sqrt @ A_used @ D_inv_sqrt
    return A_used, area, sigma


A_base, areas, _ = build_huc12_adjacency_hybrid(gdf_full)
N = len(unique_huc12s)
A_train = A_base.copy()
A_train[test_node_idx, :] = 0
A_train[:, test_node_idx] = 0
A_test = A_base.copy()
A_test[train_node_idx, :] = 0
A_test[:, train_node_idx] = 0
print(f"A_base {A_base.shape}, A_train/ A_test masked")

A_base (626, 626), A_train/ A_test masked


## Static features (with normalization)

In [14]:
def create_static_features(huc_dfs, huc_ids, static_from_model_ready, train_node_idx, areas=None):
    """Create static feature matrix aligned to `huc_ids`.

    Numeric static features are standardized using *training nodes only* (S2a strict holdout),
    then the same transform is applied to all nodes.
    """
    static_df = huc_dfs.drop_duplicates("huc_id").set_index("huc_id").reindex(huc_ids)
    train_node_idx = np.atleast_1d(np.asarray(train_node_idx, dtype=int))

    feature_list = []
    for col in static_from_model_ready:
        if col not in static_df.columns:
            raise ValueError(f"Static column '{col}' not in model-ready data.")

        s = static_df[col]
        if pd.api.types.is_numeric_dtype(s):
            x = s.to_numpy(dtype=float)
            x_train = x[train_node_idx]
            mu = np.nanmean(x_train)
            sigma = np.nanstd(x_train)
            if (not np.isfinite(sigma)) or sigma == 0:
                sigma = 1.0
            x_norm = (x - mu) / sigma
            feature_list.append(x_norm[:, None])
        else:
            # If you add categorical static features later, consider fitting categories on train only.
            onehot = pd.get_dummies(s.fillna("Unknown"), prefix=col.replace(" ", "_"))
            feature_list.append(onehot.values)

    features = np.concatenate(feature_list, axis=1)
    return torch.tensor(features, dtype=torch.float32)


static_features = create_static_features(
    huc_dfs, unique_huc12s, FINAL_STATIC_MR, train_node_idx=train_node_idx, areas=areas
)
print(f"static_features {static_features.shape}")

static_features torch.Size([626, 1])


## Train 5 models (5 seeds), log to MLflow

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
if mlflow.active_run():
    mlflow.end_run()

run_metrics = []
trained_models = []

for seed in SEEDS:
    set_seed(seed)
    with mlflow.start_run(run_name=f"seed_{seed}"):
        mlflow.log_params({
            "seed": seed,
            "n_train": n_train,
            "n_test": n_test,
            "num_epochs": NUM_EPOCHS,
            "batch_size": BATCH_SIZE,
            "seq_len": SEQ_LEN,
            "val_split": VAL_SPLIT,
            "features": ",".join(FINAL_DYNAMIC + FINAL_STATIC_MR),
        })
        model, val_metrics = train_model(
            dynamic_features=dynamic_full,
            target_tensor=target_tensor,
            adj_matrix=A_train,
            static_features=static_features,
            train_node_idx=train_node_idx,
            num_epochs=NUM_EPOCHS,
            batch_size=BATCH_SIZE,
            seq_len=SEQ_LEN,
            val_split=VAL_SPLIT,
            early_stopping=EARLY_STOPPING,
            patience=PATIENCE,
            rel_threshold=REL_THRESHOLD,
            plot=False,
            log_model=True,
        )
        run_metrics.append({"seed": seed, **val_metrics})
        trained_models.append(model)

2026/03/11 00:26:34 INFO mlflow.tracking.fluent: Experiment with name 'stgnn_final_model' does not exist. Creating a new experiment.


training model
Using device: cuda
Train samples: 11359, Val samples: 2818, Train nodes: 495


In [16]:
df_metrics = pd.DataFrame(run_metrics)
print("Val metrics per seed:")
print(df_metrics)
print(f"\nMean val_kge: {df_metrics['val_kge'].mean():.6f}, std: {df_metrics['val_kge'].std():.6f}")
print(f"Mean val_mse: {df_metrics['val_mse'].mean():.6f}, std: {df_metrics['val_mse'].std():.6f}")

Val metrics per seed:
   seed   val_mse   val_kge
0    41  0.000440  0.988551
1    42  0.000438  0.977862
2    43  0.000464  0.984238
3    44  0.000445  0.988119
4    45  0.000743  0.967568

Mean val_kge: 0.981268, std: 0.008776
Mean val_mse: 0.000506, std: 0.000133


## Test: run predictions on test HUC12s (A_test), save predictions

In [17]:
# Collect per-run predictions for test nodes
all_preds = []   # will become [n_runs, n_nodes, n_timesteps]
for i, model in enumerate(trained_models):
    preds_t, targets_t, _ = run_test_predictions(
        model, dynamic_full, target_tensor, A_test, static_features,
        test_node_idx=test_node_idx, batch_size=BATCH_SIZE, seq_len=SEQ_LEN,
    )
    all_preds.append(preds_t)
    if i == 0:
        targets_test = targets_t

all_preds = np.array(all_preds)  # [n_runs, n_nodes, n_timesteps]
preds_mean = all_preds.mean(axis=0)
preds_std = all_preds.std(axis=0)

n_runs, n_test_nodes, n_test_timesteps = all_preds.shape
test_huc12_order = [unique_huc12s[j] for j in test_node_idx]
test_dates = master_index[SEQ_LEN : SEQ_LEN + n_test_timesteps]

In [18]:
# Save detailed predictions to CSV (per run + mean + target)
rows = []
for i, huc_id in enumerate(test_huc12_order):
    for t, day in enumerate(test_dates):
        row = {
            "huc_id": huc_id,
            "day": day,
            "pred_run_1": float(all_preds[0, i, t]),
            "pred_run_2": float(all_preds[1, i, t]),
            "pred_run_3": float(all_preds[2, i, t]),
            "pred_run_4": float(all_preds[3, i, t]),
            "pred_run_5": float(all_preds[4, i, t]),
            "pred_mean": float(preds_mean[i, t]),
            "pred_std": float(preds_std[i, t]),
            "UA_swe": float(targets_test[i, t]),
        }
        rows.append(row)

df_predictions = pd.DataFrame(rows)
csv_path = "st_gnn_huc_split_test_results_all_runs.csv"
df_predictions.to_csv(csv_path, index=False)
print(f"Saved {len(df_predictions)} rows to {csv_path}")

# Save same data to Zarr for efficient analysis
zarr_path = "st_gnn_huc_split_test_results_all_runs.zarr"
root = zarr.open(zarr_path, mode="w")
root.create_dataset("preds", data=all_preds, chunks=(1, n_test_nodes, n_test_timesteps))
root.create_dataset("targets", data=targets_test, chunks=(n_test_nodes, n_test_timesteps))
root.create_dataset("huc_ids", data=np.array(test_huc12_order, dtype="U20"))
root.create_dataset("dates", data=np.array(test_dates, dtype="U20"))
print(f"Saved Zarr store to {zarr_path}")

Saved 1861117 rows to st_gnn_final_test_predictions.csv
Saved Zarr store to st_gnn_final_test_predictions.zarr


## Evaluation

> **Note (precomputed outputs in S3):** The combined “all runs” test outputs for this spatial split are stored in S3 and can be downloaded/inspected without re-running the notebook.
>
> - `s3://spatiotemporal-results/stgnn/spatial_split/st_gnn_huc_split_test_results_all_runs.csv.gz`(csv file)
> - `s3://spatiotemporal-results/stgnn/spatial_split/st_gnn_huc_split_test_results_all_runs.tar.gz` (archive containing the Zarr store)


In [19]:
# Load and inspect the saved Zarr store (no re-run of model)
zarr_path = "st_gnn_huc_split_test_results_all_runs.zarr"
root = zarr.open(zarr_path, mode="r")

print("Zarr store keys:", list(root.keys()))
print("preds shape:", root["preds"].shape, "  (n_runs, n_test_nodes, n_test_timesteps)")
print("targets shape:", root["targets"].shape, "  (n_test_nodes, n_test_timesteps)")
print("huc_ids shape:", root["huc_ids"].shape)
print("dates shape:", root["dates"].shape)
print("\nFirst 5 test HUC12s:", root["huc_ids"][:5])
print("First 5 dates:", root["dates"][:5])
print("\nPreds slice [run=0, node=0, :10]:", np.array(root["preds"][0, 0, :10]))
print("Targets slice [node=0, :10]:", np.array(root["targets"][0, :10]))

Zarr store keys: ['dates', 'huc_ids', 'preds', 'targets']
preds shape: (5, 131, 14207)   (n_runs, n_test_nodes, n_test_timesteps)
targets shape: (131, 14207)   (n_test_nodes, n_test_timesteps)
huc_ids shape: (131,)
dates shape: (14207,)

First 5 test HUC12s: ['171100110103' '171100200508' '171100130106' '171001020204'
 '171100090603']
First 5 dates: ['1983-11-07' '1983-11-08' '1983-11-09' '1983-11-10' '1983-11-11']

Preds slice [run=0, node=0, :10]: [0.10423428 0.13866752 0.11550945 0.10047287 0.13066447 0.11795765
 0.10539627 0.108715   0.12399703 0.12420171]
Targets slice [node=0, :10]: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [20]:
root

<zarr.hierarchy.Group '/' read-only>

In [ ]:
# Test KGE per run (5 values) from saved Zarr — no model re-run
root = zarr.open("st_gnn_huc_split_test_results_all_runs.zarr", mode="r")
preds = np.array(root["preds"])   # [n_runs, n_test_nodes, n_test_timesteps]
targets = np.array(root["targets"])  # [n_test_nodes, n_test_timesteps]

n_runs = preds.shape[0]
test_kge_per_run = []
for r in range(n_runs):
    y_pred = preds[r].ravel()
    y_true = targets.ravel()
    kge, _, _, _ = tef.kling_gupta_efficiency(y_true, y_pred)
    test_kge_per_run.append(kge)

test_kge_per_run = np.array(test_kge_per_run)
print("Test KGE per run (5 seeds):", test_kge_per_run)
print(f"Test KGE mean: {test_kge_per_run.mean():.6f}, std: {test_kge_per_run.std():.6f}")

# Optional: same from CSV (pred_run_1..pred_run_5 vs UA_swe)
# df = pd.read_csv("st_gnn_final_test_predictions.csv")
# for c in ["pred_run_1","pred_run_2","pred_run_3","pred_run_4","pred_run_5"]:
#     kge, _, _, _ = tef.kling_gupta_efficiency(df["UA_swe"].values, df[c].values)
#     print(c, kge)

Test KGE per run (5 seeds): [ 0.14942961  0.59532339 -1.40076029 -0.07296582 -0.10513595]
Test KGE mean: -0.166822, std: 0.665747


In [ ]:
# Test metrics per run (5 values) from saved Zarr — no model re-run
# Uses the shared compute_metrics() helper from train_eval_pipeline.

root = zarr.open("st_gnn_huc_split_test_results_all_runs.zarr", mode="r")
preds = np.array(root["preds"])      # [n_runs, n_test_nodes, n_test_timesteps]
targets = np.array(root["targets"])  # [n_test_nodes, n_test_timesteps]

n_runs = preds.shape[0]
y_true = targets.ravel()

metrics_per_run = []
for r in range(n_runs):
    y_pred = preds[r].ravel()
    metrics_per_run.append(tef.compute_metrics(y_true, y_pred))

df_test_metrics = pd.DataFrame(metrics_per_run)
print("Test metrics per run (5 seeds):")
display(df_test_metrics)

for m in ["kge", "mse", "mae", "r2"]:
    vals = df_test_metrics[m].to_numpy(dtype=float)
    print(f"Test {m} mean: {np.nanmean(vals):.6f}, std: {np.nanstd(vals):.6f}")

# Optional: same using the saved CSV (pred_run_1..pred_run_5 vs UA_swe)
# df = pd.read_csv("st_gnn_final_test_predictions.csv")
# csv_metrics = []
# for c in ["pred_run_1", "pred_run_2", "pred_run_3", "pred_run_4", "pred_run_5"]:
#     csv_metrics.append(tef.compute_metrics(df["UA_swe"].values, df[c].values))
# display(pd.DataFrame(csv_metrics))

Test metrics per run (5 seeds):


,mse,mae,r2,kge
0,0.012931,0.100466,0.767436,0.149430
1,0.016449,0.103653,0.704152,0.595323
2,0.105192,0.282405,-0.891930,-1.400760
3,0.023875,0.129763,0.570586,-0.072966
4,0.019741,0.132214,0.644955,-0.105136


Test kge mean: -0.166822, std: 0.665747
Test mse mean: 0.035637, std: 0.034965
Test mae mean: 0.149700, std: 0.067614
Test r2 mean: 0.359040, std: 0.628858
